In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
from io import StringIO


# 1) load CSV with caching
url = 'https://opendata.arcgis.com/datasets/bbb2e4f589ba40d692fab712ae37b9ac_1.csv'
cache = {}
CACHE_TTL = 3600  # 1 hour

def get_cached(key):
    entry = cache.get(key)
    if entry and time.time() - entry['time'] < CACHE_TTL:
        return entry['data']
    return None

def set_cache(key, data):
    cache[key] = {'data': data, 'time': time.time()}

cached_data = get_cached(url)
if cached_data is not None:
    df = cached_data
else:
    resp = requests.get(url)
    resp.raise_for_status()
    csv_text = resp.content.decode('utf-8-sig')
    df = pd.read_csv(StringIO(csv_text))
    df.columns = [c.lstrip('\ufeff').strip() for c in df.columns]
    set_cache(url, df)

print("Data loaded. Columns:", df.columns.tolist())

# Ensure numeric columns
for col in ['Confirmed', 'Deaths']:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(0)

# Compute Recovered = Confirmed - Deaths
df['Recovered'] = (df['Confirmed'] - df['Deaths']).clip(lower=0).astype(int)

# Define region column for use in cells below
region_col = 'Country_Region'


# Import

- ## Het code block laad de api in en laat dan de columns zien voor mijn gemak, het is voornamelijk bedoeld voor de caching.

In [ ]:
agg_deaths = df.groupby(region_col, as_index=False).agg({'Deaths': 'sum'})
top = agg_deaths.sort_values(by='Deaths', ascending=False).head(10)
labels = top[region_col].tolist()
sizes = top['Deaths'].tolist()

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=None,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct/100.0*sum(sizes))):,})",
    startangle=140,
    pctdistance=0.78,
    colors=plt.get_cmap('tab20').colors
)
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax.add_artist(centre_circle)
ax.legend(wedges, labels, title='Country / Region', loc='center left', bbox_to_anchor=(1, 0, 0.5, 1))
ax.set_title('COVID-19 Top 10 Deaths by Country / Region')
ax.axis('equal')
plt.tight_layout()
plt.show()


# Donut chart

- ## het code block maakt een donut chart van de top 10 landen van doden

In [ ]:
# Search + filtered visualization + interactive map
search = input("Enter region name to search in `Country_Region` (case-insensitive, leave blank for all): ").strip()

# Filter by search (exact first, then substring)
if search:
    qlow = search.lower()
    low = df[region_col].astype(str).str.lower()
    filtered = df[low == qlow]  # exact match
    if filtered.empty:
        filtered = df[low.str.contains(qlow, na=False)]  # substring fallback
else:
    filtered = df

if filtered.empty:
    raise ValueError(f"No rows match '{search}' in column `{region_col}`.")

# Aggregate by region
agg = filtered.groupby(region_col, as_index=False).agg({'Confirmed': 'sum', 'Recovered': 'sum', 'Deaths': 'sum'})
agg = agg.sort_values(by='Deaths', ascending=False)

# Plot stacked bar (only filtered results)
if len(agg) > 1:
    plot_data = agg.head(20)
    figsize = (12, 6)
else:
    plot_data = agg
    figsize = (8, 4)

plot_df = plot_data.set_index(region_col)[['Confirmed', 'Recovered', 'Deaths']]
ax = plot_df.plot(kind='bar', stacked=False, figsize=figsize, color=['#6baed6', '#74c476', '#de2d26'])
ax.set_title(f'COVID-19 - Confirmed / Recovered / Deaths{" (filtered)" if search else ""}')
ax.set_xlabel('Country / Region')
ax.set_ylabel('Counts')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Visualisaties - Top 10 Bar Chart, Bubble Map en Time Series

try:
    import plotly.express as px
    _HAS_PLOTLY = True
except ImportError:
    px = None
    _HAS_PLOTLY = False

# Vraag welk land voor time series
country_ts = input("Enter country name for time series (leave blank for global): ").strip()

# 1) Global Top 10 bar chart - sorted descending (high to low)
agg_all = df.groupby('Country_Region', as_index=False).agg({
    'Confirmed': 'sum',
    'Recovered': 'sum',
    'Deaths': 'sum'
}).sort_values('Deaths', ascending=False).head(10)

plot_df = agg_all.set_index('Country_Region')[['Confirmed', 'Recovered', 'Deaths']]
ax = plot_df.plot(kind='bar', stacked=False, figsize=(12, 6), color=['#6baed6', '#74c476', '#de2d26'])
ax.set_title('COVID-19 - Top 10')
ax.set_xlabel('Country / Region')
ax.set_ylabel('Counts')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.6)
plt.tight_layout()
plt.show()

# 2) Choropleth Map - Globale kaart met landgrenzen (geen country filter)
if _HAS_PLOTLY:
    try:
        agg_bubble = df.groupby('Country_Region', as_index=False).agg({
            'Confirmed': 'sum',
            'Recovered': 'sum',
            'Deaths': 'sum'
        })

        fig = px.choropleth(
            agg_bubble,
            locations='Country_Region',
            locationmode='country names',
            color='Confirmed',
            hover_name='Country_Region',
            hover_data={'Confirmed': ':.0f', 'Recovered': ':.0f', 'Deaths': ':.0f', 'Country_Region': False},
            color_continuous_scale='Blues',
            title='COVID-19 - Confirmed Cases by Country (Global)',
        )
        fig.update_layout(coloraxis_colorbar=dict(title='Confirmed'))
        fig.show()
    except Exception as e:
        print('map failed:', e)
else:
    print('Plotly not installed - bubble map skipped')

# 3) Time Series - Country specific of Global
try:
    # Filter by country if provided
    if country_ts:
        qlow = country_ts.lower()
        ts_data = df[df['Country_Region'].astype(str).str.lower().str.contains(qlow, na=False)]
        if ts_data.empty:
            print(f"⚠ Geen gegevens gevonden voor '{country_ts}'")
            ts_data = df
            country_ts = ''
    else:
        ts_data = df

    df_ts = ts_data.copy()
    df_ts['Last_Update'] = pd.to_datetime(df_ts['Last_Update'], errors='coerce')
    df_ts = df_ts.dropna(subset=['Last_Update'])

    ts = df_ts.groupby('Last_Update').agg({
        'Confirmed': 'sum',
        'Recovered': 'sum',
        'Deaths': 'sum'
    }).sort_index()

    if ts.empty:
        print('Geen geldige datagegevens voor time series')
    else:
        title_suffix = f' ({country_ts})' if country_ts else ' (Global)'

        if _HAS_PLOTLY:
            fig = px.line(
                ts.reset_index(),
                x='Last_Update',
                y=['Confirmed', 'Recovered', 'Deaths'],
                title=f'COVID-19 - Time Series{title_suffix}',
                labels={'Last_Update': 'Date', 'value': 'Count'}
            )
            fig.update_traces(mode='lines+markers')
            fig.show()
        else:
            plt.figure(figsize=(12, 6))
            plt.plot(ts.index, ts['Confirmed'], label='Confirmed', marker='o', color='#6baed6')
            plt.plot(ts.index, ts['Recovered'], label='Recovered', marker='o', color='#74c476')
            plt.plot(ts.index, ts['Deaths'], label='Deaths', marker='o', color='#de2d26')
            plt.title(f'COVID-19 - Time Series{title_suffix}')
            plt.xlabel('Date')
            plt.ylabel('Counts')
            plt.legend()
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
except Exception as e:
    print('Time series failed:', e)
